In [ ]:
#RNN

In [5]:
#Splitting the train data into train, test, valditation (80,10,10)

from pathlib import Path
import pandas as pd, ast
from sklearn.model_selection import train_test_split


df = pd.read_csv(Path("..") / "data" / "SentiMix" / "train.csv")
df["text"]  = df["sentence"].apply(lambda s: " ".join(ast.literal_eval(s)))
df["label"] = df["sentiment"]
df = df[["id", "text", "label"]] # Drop all column except "id","text","label"
# print(df.head(5))
print("empty text rows:", (df["text"].str.strip() == "").sum())
print("duplicate ids:", df["id"].duplicated().sum())

df = df[df["text"].str.strip() != ""].drop_duplicates(subset="id")

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42)

print(len(train_df), len(val_df), len(test_df))
print(train_df["label"].value_counts(normalize=True))

empty text rows: 0
duplicate ids: 2
11128 1391 1391
label
neutral     0.375539
positive    0.330607
negative    0.293853
Name: proportion, dtype: float64


In [3]:
import torch
import torch.nn as nn
from collections import Counter
from torch.utils.data import TensorDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

label2id = {"negative": 0, "neutral": 1, "positive": 2}
for df in [train_df, val_df, test_df]:
    df["label_id"] = df["label"].str.lower().str.strip().map(label2id)

def tokenize(text):
    return str(text).lower().split()

# Creating Vocabulary
counter = Counter()
for t in train_df["text"]:
    counter.update(tokenize(t))

vocab = {"<pad>": 0, "<unk>": 1}
for word in counter:
    vocab[word] = len(vocab)
print("Vocab size:", len(vocab))

MAX_LEN = 56

def encode(text):
    ids = [vocab.get(w, 1) for w in tokenize(text)][:MAX_LEN]   # unseen word -> 1
    length = max(len(ids), 1)
    ids += [0] * (MAX_LEN - len(ids))                           # pad with 0
    return ids, length

def make_loader(df, shuffle):
    encoded = [encode(t) for t in df["text"]]
    X = torch.tensor([e[0] for e in encoded])
    lengths = torch.tensor([e[1] for e in encoded])
    y = torch.tensor(df["label_id"].values)
    return DataLoader(TensorDataset(X, lengths, y), batch_size=64, shuffle=shuffle)

train_loader = make_loader(train_df, True)
val_loader   = make_loader(val_df, False)
test_loader  = make_loader(test_df, False)

xb, lb, yb = next(iter(train_loader))
print(xb.shape, lb.shape, yb.shape)   # expect [64, 56] [64] [64]

Vocab size: 42554
torch.Size([64, 56]) torch.Size([64]) torch.Size([64])


In [4]:
class RecurrentClassifier(nn.Module):
    def __init__(self, vocab_size, rnn_type="lstm", bidirectional=False,
                 embed_dim=100, hidden_dim=128, num_classes=3, dropout=0.3):
        super().__init__()
        self.rnn_type = rnn_type
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout)
        rnn_cls = {"rnn": nn.RNN, "lstm": nn.LSTM}[rnn_type]
        self.rnn = rnn_cls(embed_dim, hidden_dim, batch_first=True, bidirectional=bidirectional)
        self.fc = nn.Linear(hidden_dim * (2 if bidirectional else 1), num_classes)

    def forward(self, x, lengths):
        emb = self.dropout(self.embedding(x))
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, h = self.rnn(packed)
        if self.rnn_type == "lstm":
            h = h[0]
        if self.bidirectional:
            h = torch.cat([h[-2], h[-1]], dim=1)
        else:
            h = h[-1]
        return self.fc(self.dropout(h))

4285227 trainable parameters
torch.Size([64, 3])


In [5]:
import copy
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 10

def evaluate(model, loader):
    model.eval()
    preds, labels, total_loss = [], [], 0
    with torch.no_grad():
        for x, lengths, y in loader:
            x, lengths, y = x.to(device), lengths.to(device), y.to(device)
            logits = model(x, lengths)
            total_loss += criterion(logits, y).item() * len(y)
            preds.extend(logits.argmax(1).cpu().tolist())
            labels.extend(y.cpu().tolist())
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    micro = precision_recall_fscore_support(labels, preds, average="micro", zero_division=0)[2]
    metrics = {"loss": total_loss / len(labels), "acc": accuracy_score(labels, preds),
               "precision_w": p, "recall_w": r, "f1_w": f1, "f1_micro": micro}
    return metrics, labels, preds

best_f1, best_state = 0, None

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0
    for x, lengths, y in train_loader:
        x, lengths, y = x.to(device), lengths.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x, lengths), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running += loss.item() * len(y)

    train_loss = running / len(train_loader.dataset)
    val, _, _ = evaluate(model, val_loader)
    print(f"Epoch {epoch}: train_loss={train_loss:.3f} val_loss={val['loss']:.3f} "
          f"acc={val['acc']:.3f} f1_w={val['f1_w']:.3f} f1_micro={val['f1_micro']:.3f}")

    if val["f1_w"] > best_f1:
        best_f1 = val["f1_w"]
        best_state = copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)
test, labels, preds = evaluate(model, test_loader)
print("TEST:", test)
print(classification_report(labels, preds, target_names=["negative", "neutral", "positive"], zero_division=0))

Epoch 1: train_loss=1.088 val_loss=1.039 acc=0.437 f1_w=0.389 f1_micro=0.437
Epoch 2: train_loss=0.975 val_loss=0.942 acc=0.537 f1_w=0.523 f1_micro=0.537
Epoch 3: train_loss=0.895 val_loss=0.930 acc=0.532 f1_w=0.510 f1_micro=0.532
Epoch 4: train_loss=0.815 val_loss=0.953 acc=0.503 f1_w=0.496 f1_micro=0.503
Epoch 5: train_loss=0.714 val_loss=0.967 acc=0.546 f1_w=0.543 f1_micro=0.546
Epoch 6: train_loss=0.593 val_loss=1.031 acc=0.551 f1_w=0.548 f1_micro=0.551
Epoch 7: train_loss=0.479 val_loss=1.082 acc=0.541 f1_w=0.528 f1_micro=0.541
Epoch 8: train_loss=0.376 val_loss=1.375 acc=0.529 f1_w=0.504 f1_micro=0.529
Epoch 9: train_loss=0.283 val_loss=1.424 acc=0.508 f1_w=0.511 f1_micro=0.508
Epoch 10: train_loss=0.204 val_loss=1.509 acc=0.505 f1_w=0.508 f1_micro=0.505
TEST: {'loss': 1.0630714500833123, 'acc': 0.5463695183321352, 'precision_w': 0.5413602999265292, 'recall_w': 0.5463695183321352, 'f1_w': 0.542559144969602, 'f1_micro': 0.5463695183321352}
              precision    recall  f1-sco